<a href="https://colab.research.google.com/github/muzaqqa/Adept-Internship/blob/main/Multilingual_Learning_Path_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install fastapi==0.111.0 uvicorn==0.30.3 gradio==4.44.0 \
               langchain==0.2.5 langchain-community==0.2.5 langgraph==0.2.14 \
               python-dotenv==1.0.1 httpx==0.27.2 pydantic==2.8.2 \
               loguru==0.7.2 orjson==3.10.7 \
               google-api-python-client==2.137.0 google-auth==2.33.0 google-auth-oauthlib==1.2.1 \
               faiss-cpu==1.8.0.post1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.6/974.6 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.9/423.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/1

In [2]:
import os, textwrap, json
from pathlib import Path

# Colab-safe project dir
PROJECT_DIR = Path("/content/ml_learning_path")
os.makedirs(PROJECT_DIR, exist_ok=True)

# Write a .env template if not present
env_path = PROJECT_DIR / ".env"
if not env_path.exists():
    env_path.write_text(textwrap.dedent("""
        # --- Optional observability ---
        LANGSMITH_API_KEY=
        LANGSMITH_TRACING=false
        LANGSMITH_PROJECT=learning-path-mvp

        LANGFUSE_PUBLIC_KEY=
        LANGFUSE_SECRET_KEY=
        LANGFUSE_HOST=

        # --- Optional Google Classroom OAuth (if you have credentials.json uploaded) ---
        GOOGLE_CLASSROOM_ENABLED=false

        # --- External service endpoints (safe defaults) ---
        LIBRETRANSLATE_URL=https://libretranslate.com/translate
        EDX_CATALOG_URL=https://raw.githubusercontent.com/openedx/openedx-courses/refs/heads/main/demo_catalog.json

        # --- App settings ---
        APP_HOST=0.0.0.0
        APP_PORT=8000
        """).strip())
print(f".env located at: {env_path}")


.env located at: /content/ml_learning_path/.env


In [3]:
import os, threading, time
from typing import List, Optional, Dict, Any
from dotenv import load_dotenv
from loguru import logger
import orjson

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import httpx

load_dotenv(os.fspath(PROJECT_DIR / ".env"))

# -------- Structured logging (JSON) --------
def jlog(event: str, **k):
    logger.bind(event=event).info(orjson.dumps(k).decode())

# -------- Models --------
class Course(BaseModel):
    provider: str
    id: str
    title: str
    url: str
    language: str = "en"
    level: Optional[str] = None
    topics: List[str] = []

class PlanRequest(BaseModel):
    goal: str = Field(..., description="e.g., 'Learn Computer Vision basics in 4 weeks'")
    target_language: str = "en"
    weeks: int = 4
    hours_per_week: int = 5
    preferred_provider: Optional[str] = None  # 'edx' or 'coursera'
    level: Optional[str] = None               # 'beginner'/'intermediate'/'advanced'

class QuizQuestion(BaseModel):
    question: str
    options: List[str]
    answer_index: int

class ProgressEvent(BaseModel):
    user_id: str
    course_id: str
    completed_percent: float

class CatalogQuery(BaseModel):
    text: str
    provider: Optional[str] = None
    language: Optional[str] = None
    level: Optional[str] = None

# -------- External Adapters --------
class CatalogAdapter:
    async def search(self, text:str, language:Optional[str], level:Optional[str]) -> List[Course]:
        raise NotImplementedError

class EdxAdapter(CatalogAdapter):
    def __init__(self, url: str):
        self.url = url
    async def search(self, text, language, level):
        # Public demo JSON (shape: list of {title,url,language,level,topics})
        async with httpx.AsyncClient(timeout=20) as client:
            r = await client.get(self.url)
            r.raise_for_status()
            data = r.json()
        results = []
        for i, c in enumerate(data[:300]):  # cap for speed
            if text.lower() not in (c.get("title","")+ " " + " ".join(c.get("topics",[]))).lower():
                continue
            if language and c.get("language","en").lower()!=language.lower():
                continue
            if level and (c.get("level") or "").lower()!=level.lower():
                continue
            results.append(Course(
                provider="edx",
                id=f"edx_{i}",
                title=c.get("title","Untitled"),
                url=c.get("url",""),
                language=c.get("language","en"),
                level=c.get("level"), topics=c.get("topics",[])
            ))
        jlog("edx_search", q=text, n=len(results))
        return results

class CourseraAdapter(CatalogAdapter):
    # Coursera open catalog needs partner auth; we provide a graceful mock
    async def search(self, text, language, level):
        mock = [
            Course(provider="coursera", id="c_mock_ml", title="Machine Learning Specialization",
                   url="https://www.coursera.org/specializations/machine-learning-introduction",
                   language="en", level="beginner", topics=["ml","ai"]),
            Course(provider="coursera", id="c_mock_cv", title="Computer Vision Basics",
                   url="https://www.coursera.org/learn/computer-vision-basics",
                   language="en", level="beginner", topics=["vision","cv"])
        ]
        results = [c for c in mock if text.lower() in c.title.lower()]
        if language: results = [c for c in results if c.language.lower()==language.lower()]
        if level: results = [c for c in results if (c.level or "").lower()==level.lower()]
        jlog("coursera_mock_search", q=text, n=len(results))
        return results

# -------- Translation (LibreTranslate public endpoint) --------
async def translate_text(text:str, target_lang:str) -> str:
    url = os.getenv("LIBRETRANSLATE_URL","https://libretranslate.com/translate")
    payload = {"q": text, "source": "auto", "target": target_lang}
    try:
        async with httpx.AsyncClient(timeout=20) as client:
            r = await client.post(url, data=payload, headers={"accept":"application/json"})
            r.raise_for_status()
            out = r.json()
            return out.get("translatedText", text)
    except Exception as e:
        jlog("translate_fallback", error=str(e))
        return text  # graceful fallback

# -------- Tiny "agent" using LangChain tools --------
from langchain.agents import Tool, initialize_agent
from langchain.llms.fake import FakeListLLM  # offline tiny LLM replacement for control logic

# Our tools: catalog search & translation
edx = EdxAdapter(os.getenv("EDX_CATALOG_URL"))
coursera = CourseraAdapter()

async def tool_search_catalog(q:str, provider:Optional[str]=None, language:Optional[str]=None, level:Optional[str]=None):
    if provider=="coursera":
        return [c.model_dump() for c in await coursera.search(q, language, level)]
    # default try edX, then coursera mock
    results = await edx.search(q, language, level)
    if not results:
        results = await coursera.search(q, language, level)
    return [c.model_dump() for c in results]

async def tool_translate(text:str, target_lang:str):
    return await translate_text(text, target_lang)

tools = [
    Tool(name="catalog_search", func=lambda q: httpx.run(tool_search_catalog(q)),
         description="Search MOOC catalog for courses relevant to a text query."),
    Tool(name="translate", func=lambda args: httpx.run(tool_translate(args['text'], args['target_lang'])) if isinstance(args, dict) else args,
         description="Translate text to target language, args={'text','target_lang'}."),
]
# We use FakeListLLM to keep it offline and deterministic. Prompts are minimal.
llm = FakeListLLM(responses=["SEARCH", "TRANSLATE", "DONE"])

# NOTE: We'll not rely on actual LLM reasoning; the 'agent' here orchestrates simple steps.

# -------- Plan builder --------
def build_weekly_plan(courses: List[Course], weeks:int, hours_per_week:int, language:str):
    # simple even split: each week get a course module (pretend 4 modules/course)
    plan = []
    week = 1
    for c in courses[:min(len(courses), weeks)]:
        plan.append({
            "week": week,
            "course_title": c.title,
            "provider": c.provider,
            "url": c.url,
            "language": language,
            "target_hours": hours_per_week,
            "milestones": [f"Complete 25% of {c.title}", f"Attempt practice quiz"]
        })
        week += 1
    return plan

# -------- FastAPI app --------
app = FastAPI(title="Learning Path Generator", version="0.1.0")

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/catalog/search")
async def catalog_search(q: CatalogQuery):
    items = await tool_search_catalog(q.text, q.provider, q.language, q.level)
    return {"results": items}

@app.post("/plan/generate")
async def generate_plan(req: PlanRequest):
    items = await tool_search_catalog(req.goal, req.preferred_provider, "en", req.level)
    courses = [Course(**it) for it in items][:max(1, min(5, req.weeks))]
    if not courses:
        raise HTTPException(404, "No courses found for your goal.")
    # translate titles if needed
    if req.target_language and req.target_language.lower() != "en":
        for c in courses:
            c.title = await translate_text(c.title, req.target_language)
    plan = build_weekly_plan(courses, req.weeks, req.hours_per_week, req.target_language)
    jlog("plan_generated", weeks=req.weeks, n=len(courses))
    return {"plan": plan}

@app.post("/quiz/generate")
def generate_quiz(topic: str):
    # ultra-simple static quiz generator (stub)
    qs = [
        QuizQuestion(question=f"{topic}: What is supervised learning?",
                     options=["Clustering unlabeled data","Learning from labeled examples","Random guessing","Data compression"],
                     answer_index=1),
        QuizQuestion(question=f"{topic}: Which is an optimizer?",
                     options=["ReLU","Adam","Dropout","Softmax"],
                     answer_index=1),
    ]
    return {"questions": [q.model_dump() for q in qs]}

# progress tracking (in-memory)
PROGRESS_DB: Dict[str, Dict[str, float]] = {}

@app.post("/progress/update")
def progress_update(ev: ProgressEvent):
    user = PROGRESS_DB.setdefault(ev.user_id, {})
    user[ev.course_id] = ev.completed_percent
    jlog("progress_update", user_id=ev.user_id, course_id=ev.course_id, pct=ev.completed_percent)
    return {"ok": True, "user_progress": user}

# Google Classroom (optional)
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials

@app.post("/classroom/sync")
def classroom_sync(user_id: str, title: str, description: str="Learning plan task"):
    enabled = os.getenv("GOOGLE_CLASSROOM_ENABLED","false").lower()=="true"
    if not enabled:
        return {"ok": False, "reason": "Google Classroom disabled (set GOOGLE_CLASSROOM_ENABLED=true and provide creds)."}
    try:
        creds_path = str((PROJECT_DIR/"token.json"))
        if not os.path.exists(creds_path):
            return {"ok": False, "reason": "No OAuth token.json present."}
        creds = Credentials.from_authorized_user_file(creds_path,
                                                      scopes=["https://www.googleapis.com/auth/classroom.courses","https://www.googleapis.com/auth/classroom.coursework.me"])
        service = build("classroom", "v1", credentials=creds, cache_discovery=False)
        # Minimal demo: list first course
        courses = service.courses().list(pageSize=1).execute().get("courses", [])
        course_id = courses[0]["id"] if courses else None
        if not course_id:
            return {"ok": False, "reason": "No Classroom courses found."}
        body = {"title": title, "description": description, "workType": "ASSIGNMENT", "state": "PUBLISHED"}
        created = service.courses().courseWork().create(courseId=course_id, body=body).execute()
        return {"ok": True, "courseWorkId": created.get("id")}
    except Exception as e:
        return {"ok": False, "error": str(e)}


In [4]:
import os, threading, time
from typing import List, Optional, Dict, Any
from dotenv import load_dotenv
from loguru import logger
import orjson

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import httpx

load_dotenv(os.fspath(PROJECT_DIR / ".env"))

# -------- Structured logging (JSON) --------
def jlog(event: str, **k):
    logger.bind(event=event).info(orjson.dumps(k).decode())

# -------- Models --------
class Course(BaseModel):
    provider: str
    id: str
    title: str
    url: str
    language: str = "en"
    level: Optional[str] = None
    topics: List[str] = []

class PlanRequest(BaseModel):
    goal: str = Field(..., description="e.g., 'Learn Computer Vision basics in 4 weeks'")
    target_language: str = "en"
    weeks: int = 4
    hours_per_week: int = 5
    preferred_provider: Optional[str] = None  # 'edx' or 'coursera'
    level: Optional[str] = None               # 'beginner'/'intermediate'/'advanced'

class QuizQuestion(BaseModel):
    question: str
    options: List[str]
    answer_index: int

class ProgressEvent(BaseModel):
    user_id: str
    course_id: str
    completed_percent: float

class CatalogQuery(BaseModel):
    text: str
    provider: Optional[str] = None
    language: Optional[str] = None
    level: Optional[str] = None

# -------- External Adapters --------
class CatalogAdapter:
    async def search(self, text:str, language:Optional[str], level:Optional[str]) -> List[Course]:
        raise NotImplementedError

class EdxAdapter(CatalogAdapter):
    def __init__(self, url: str):
        self.url = url
    async def search(self, text, language, level):
        # Public demo JSON (shape: list of {title,url,language,level,topics})
        async with httpx.AsyncClient(timeout=20) as client:
            r = await client.get(self.url)
            r.raise_for_status()
            data = r.json()
        results = []
        for i, c in enumerate(data[:300]):  # cap for speed
            if text.lower() not in (c.get("title","")+ " " + " ".join(c.get("topics",[]))).lower():
                continue
            if language and c.get("language","en").lower()!=language.lower():
                continue
            if level and (c.get("level") or "").lower()!=level.lower():
                continue
            results.append(Course(
                provider="edx",
                id=f"edx_{i}",
                title=c.get("title","Untitled"),
                url=c.get("url",""),
                language=c.get("language","en"),
                level=c.get("level"), topics=c.get("topics",[])
            ))
        jlog("edx_search", q=text, n=len(results))
        return results

class CourseraAdapter(CatalogAdapter):
    # Coursera open catalog needs partner auth; we provide a graceful mock
    async def search(self, text, language, level):
        mock = [
            Course(provider="coursera", id="c_mock_ml", title="Machine Learning Specialization",
                   url="https://www.coursera.org/specializations/machine-learning-introduction",
                   language="en", level="beginner", topics=["ml","ai"]),
            Course(provider="coursera", id="c_mock_cv", title="Computer Vision Basics",
                   url="https://www.coursera.org/learn/computer-vision-basics",
                   language="en", level="beginner", topics=["vision","cv"])
        ]
        results = [c for c in mock if text.lower() in c.title.lower()]
        if language: results = [c for c in results if c.language.lower()==language.lower()]
        if level: results = [c for c in results if (c.level or "").lower()==level.lower()]
        jlog("coursera_mock_search", q=text, n=len(results))
        return results

# -------- Translation (LibreTranslate public endpoint) --------
async def translate_text(text:str, target_lang:str) -> str:
    url = os.getenv("LIBRETRANSLATE_URL","https://libretranslate.com/translate")
    payload = {"q": text, "source": "auto", "target": target_lang}
    try:
        async with httpx.AsyncClient(timeout=20) as client:
            r = await client.post(url, data=payload, headers={"accept":"application/json"})
            r.raise_for_status()
            out = r.json()
            return out.get("translatedText", text)
    except Exception as e:
        jlog("translate_fallback", error=str(e))
        return text  # graceful fallback

# -------- Tiny "agent" using LangChain tools --------
from langchain.agents import Tool, initialize_agent
from langchain.llms.fake import FakeListLLM  # offline tiny LLM replacement for control logic

# Our tools: catalog search & translation
edx = EdxAdapter(os.getenv("EDX_CATALOG_URL"))
coursera = CourseraAdapter()

async def tool_search_catalog(q:str, provider:Optional[str]=None, language:Optional[str]=None, level:Optional[str]=None):
    if provider=="coursera":
        return [c.model_dump() for c in await coursera.search(q, language, level)]
    # default try edX, then coursera mock
    results = await edx.search(q, language, level)
    if not results:
        results = await coursera.search(q, language, level)
    return [c.model_dump() for c in results]

async def tool_translate(text:str, target_lang:str):
    return await translate_text(text, target_lang)

tools = [
    Tool(name="catalog_search", func=lambda q: httpx.run(tool_search_catalog(q)),
         description="Search MOOC catalog for courses relevant to a text query."),
    Tool(name="translate", func=lambda args: httpx.run(tool_translate(args['text'], args['target_lang'])) if isinstance(args, dict) else args,
         description="Translate text to target language, args={'text','target_lang'}."),
]
# We use FakeListLLM to keep it offline and deterministic. Prompts are minimal.
llm = FakeListLLM(responses=["SEARCH", "TRANSLATE", "DONE"])

# NOTE: We'll not rely on actual LLM reasoning; the 'agent' here orchestrates simple steps.

# -------- Plan builder --------
def build_weekly_plan(courses: List[Course], weeks:int, hours_per_week:int, language:str):
    # simple even split: each week get a course module (pretend 4 modules/course)
    plan = []
    week = 1
    for c in courses[:min(len(courses), weeks)]:
        plan.append({
            "week": week,
            "course_title": c.title,
            "provider": c.provider,
            "url": c.url,
            "language": language,
            "target_hours": hours_per_week,
            "milestones": [f"Complete 25% of {c.title}", f"Attempt practice quiz"]
        })
        week += 1
    return plan

# -------- FastAPI app --------
app = FastAPI(title="Learning Path Generator", version="0.1.0")

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/catalog/search")
async def catalog_search(q: CatalogQuery):
    items = await tool_search_catalog(q.text, q.provider, q.language, q.level)
    return {"results": items}

@app.post("/plan/generate")
async def generate_plan(req: PlanRequest):
    items = await tool_search_catalog(req.goal, req.preferred_provider, "en", req.level)
    courses = [Course(**it) for it in items][:max(1, min(5, req.weeks))]
    if not courses:
        raise HTTPException(404, "No courses found for your goal.")
    # translate titles if needed
    if req.target_language and req.target_language.lower() != "en":
        for c in courses:
            c.title = await translate_text(c.title, req.target_language)
    plan = build_weekly_plan(courses, req.weeks, req.hours_per_week, req.target_language)
    jlog("plan_generated", weeks=req.weeks, n=len(courses))
    return {"plan": plan}

@app.post("/quiz/generate")
def generate_quiz(topic: str):
    # ultra-simple static quiz generator (stub)
    qs = [
        QuizQuestion(question=f"{topic}: What is supervised learning?",
                     options=["Clustering unlabeled data","Learning from labeled examples","Random guessing","Data compression"],
                     answer_index=1),
        QuizQuestion(question=f"{topic}: Which is an optimizer?",
                     options=["ReLU","Adam","Dropout","Softmax"],
                     answer_index=1),
    ]
    return {"questions": [q.model_dump() for q in qs]}

# progress tracking (in-memory)
PROGRESS_DB: Dict[str, Dict[str, float]] = {}

@app.post("/progress/update")
def progress_update(ev: ProgressEvent):
    user = PROGRESS_DB.setdefault(ev.user_id, {})
    user[ev.course_id] = ev.completed_percent
    jlog("progress_update", user_id=ev.user_id, course_id=ev.course_id, pct=ev.completed_percent)
    return {"ok": True, "user_progress": user}

# Google Classroom (optional)
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials

@app.post("/classroom/sync")
def classroom_sync(user_id: str, title: str, description: str="Learning plan task"):
    enabled = os.getenv("GOOGLE_CLASSROOM_ENABLED","false").lower()=="true"
    if not enabled:
        return {"ok": False, "reason": "Google Classroom disabled (set GOOGLE_CLASSROOM_ENABLED=true and provide creds)."}
    try:
        creds_path = str((PROJECT_DIR/"token.json"))
        if not os.path.exists(creds_path):
            return {"ok": False, "reason": "No OAuth token.json present."}
        creds = Credentials.from_authorized_user_file(creds_path,
                                                      scopes=["https://www.googleapis.com/auth/classroom.courses","https://www.googleapis.com/auth/classroom.coursework.me"])
        service = build("classroom", "v1", credentials=creds, cache_discovery=False)
        # Minimal demo: list first course
        courses = service.courses().list(pageSize=1).execute().get("courses", [])
        course_id = courses[0]["id"] if courses else None
        if not course_id:
            return {"ok": False, "reason": "No Classroom courses found."}
        body = {"title": title, "description": description, "workType": "ASSIGNMENT", "state": "PUBLISHED"}
        created = service.courses().courseWork().create(courseId=course_id, body=body).execute()
        return {"ok": True, "courseWorkId": created.get("id")}
    except Exception as e:
        return {"ok": False, "error": str(e)}


In [5]:
import uvicorn, nest_asyncio, asyncio
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host=os.getenv("APP_HOST","0.0.0.0"), port=int(os.getenv("APP_PORT","8000")), log_level="warning")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(1.5)
print("FastAPI running at http://127.0.0.1:8000/health")


Exception in thread Thread-3 (run_api):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-2167251795.py", line 5, in run_api
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 577, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 65, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 26, in run
    loop = asyncio.get_event_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 40, in _get_event_loop
    loop = events.get_event_loop_policy().get_event_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/

FastAPI running at http://127.0.0.1:8000/health


In [6]:
import gradio as gr
import httpx
BASE = "http://127.0.0.1:8000"

async def ui_generate(goal, target_lang, weeks, hours_per_week, provider, level):
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(f"{BASE}/plan/generate", json={
            "goal": goal, "target_language": target_lang, "weeks": int(weeks),
            "hours_per_week": int(hours_per_week), "preferred_provider": (provider or None), "level": (level or None)
        })
        if r.status_code != 200:
            return [], f"Error: {r.text}"
        plan = r.json()["plan"]
        rows = [[p["week"], p["course_title"], p["provider"], p["url"], p["language"], p["target_hours"], ", ".join(p["milestones"])] for p in plan]
        return rows, f"Generated plan with {len(rows)} weeks ✅"

async def ui_search(q, provider, language, level):
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(f"{BASE}/catalog/search", json={
            "text": q, "provider": provider or None, "language": language or None, "level": level or None
        })
        res = r.json()["results"]
        rows = [[c["provider"], c["title"], c["language"], c.get("level",""), c["url"]] for c in res][:50]
        return rows

async def ui_quiz(topic):
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(f"{BASE}/quiz/generate", params={"topic": topic})
        qs = r.json()["questions"]
        return json.dumps(qs, indent=2, ensure_ascii=False)

with gr.Blocks(title="Multilingual Learning Path Generator") as demo:
    gr.Markdown("## 🌍 Multilingual Learning Path Generator (MVP)\nBuild study plans from MOOC catalogs, translate to your language, track progress, and (optionally) sync to Google Classroom.")
    with gr.Tab("Generate Plan"):
        goal = gr.Textbox(label="Your learning goal", value="Learn computer vision basics")
        with gr.Row():
            lang = gr.Dropdown(["en","es","fr","de","ar","ur","zh","hi"], value="en", label="Target language")
            weeks = gr.Number(label="Weeks", value=4, precision=0)
            hpw = gr.Number(label="Hours/week", value=5, precision=0)
        with gr.Row():
            provider = gr.Dropdown(["edx","coursera",""], value="", label="Preferred provider (optional)")
            level = gr.Dropdown(["beginner","intermediate","advanced",""], value="", label="Level (optional)")
        gen_btn = gr.Button("Generate")
        plan_df = gr.Dataframe(headers=["Week","Course Title","Provider","URL","Language","Target Hours","Milestones"], interactive=False, wrap=True)
        gen_msg = gr.Markdown()
        gen_btn.click(ui_generate, [goal, lang, weeks, hpw, provider, level], [plan_df, gen_msg])

    with gr.Tab("Search Catalog"):
        q = gr.Textbox(label="Query", value="machine learning")
        with gr.Row():
            p2 = gr.Dropdown(["edx","coursera",""], value="", label="Provider")
            l2 = gr.Dropdown(["","en","es","fr","de","ar","ur","zh","hi"], value="", label="Language")
            lev2 = gr.Dropdown(["","beginner","intermediate","advanced"], value="", label="Level")
        search_btn = gr.Button("Search")
        results_df = gr.Dataframe(headers=["Provider","Title","Lang","Level","URL"], interactive=False, wrap=True)
        search_btn.click(ui_search, [q, p2, l2, lev2], [results_df])

    with gr.Tab("Quick Quiz"):
        topic = gr.Textbox(label="Topic", value="Supervised learning")
        quiz_btn = gr.Button("Generate Quiz")
        quiz_json = gr.Code(language="json")
        quiz_btn.click(ui_quiz, [topic], [quiz_json])

demo.launch(share=True)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
from collections import deque
import time

METRICS = {"plans_generated": 0, "catalog_searches": 0, "quiz_generated": 0}
LOGS = deque(maxlen=200)

# Hook loguru to also push into LOGS
logger.remove()
logger.add(lambda msg: LOGS.append(msg), level="INFO")

# Tiny endpoint to expose metrics
@app.get("/metrics")
def get_metrics():
    return {"ts": time.time(), "metrics": METRICS, "recent_logs": list(LOGS)[-10:]}

print("Metrics available at http://127.0.0.1:8000/metrics (after a restart cell, if needed).")


In [ ]:
import os
# These libs auto-pick env vars; we simply show how to turn them on.
if os.getenv("LANGSMITH_API_KEY") and os.getenv("LANGSMITH_TRACING","false").lower()=="true":
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    print("LangSmith tracing enabled.")
else:
    print("LangSmith tracing disabled (set LANGSMITH_API_KEY and LANGSMITH_TRACING=true).")

if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY") and os.getenv("LANGFUSE_HOST"):
    print("Langfuse can be enabled in your agent/tools where applicable.")
else:
    print("Langfuse not configured (set keys in .env if you want it).")


In [ ]:
# Upload your Google OAuth "credentials.json" to /content and run this once to create token.json.
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
import os, json

SCOPES = ["https://www.googleapis.com/auth/classroom.courses","https://www.googleapis.com/auth/classroom.coursework.me"]
cred_file = "/content/credentials.json"  # change if needed
token_path = PROJECT_DIR/"token.json"

if os.path.exists(cred_file):
    flow = InstalledAppFlow.from_client_secrets_file(cred_file, SCOPES)
    creds = flow.run_console()
    with open(token_path, "w") as f:
        f.write(creds.to_json())
    print("Google Classroom token saved. Set GOOGLE_CLASSROOM_ENABLED=true in .env")
else:
    print("Upload credentials.json first to enable Classroom.")


In [ ]:
# Dockerfile
FROM python:3.11-slim

WORKDIR /app
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1

COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

COPY . .
EXPOSE 8000
CMD ["uvicorn", "server.main:app", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
fastapi==0.111.0
uvicorn==0.30.3
gradio==4.44.0
langchain==0.2.5
langchain-community==0.2.5
langgraph==0.2.14
python-dotenv==1.0.1
httpx==0.27.2
pydantic==2.8.2
loguru==0.7.2
orjson==3.10.7
google-api-python-client==2.137.0
google-auth==2.33.0
google-auth-oauthlib==1.2.1
faiss-cpu==1.8.0.post1


In [ ]:
version: "3.9"
services:
  api:
    build: .
    env_file: .env
    ports: ["8000:8000"]
